In [13]:
import pandas as pd

df = pd.read_json("../data/processed/train_v8.jsonl", lines=True)
print(len(df))
df.head()

6555


,corrupted,original
0,Für Koloration von Zeichnungen oder Hilfsarbei...,Für die Koloration von Zeichnungen oder Hilfsa...
1,Wir standen nur da und schauten einander überr...,Wir standen nur da und schauten einander überr...
2,er baute die Stufenpyramide von dem Sakkara.,Er baute die Stufenpyramide von Sakkara.
3,Praktiziert wurde diese Zusammenarbeit aber be...,Praktiziert wurde diese Zusammenarbeit aber be...
4,"So lange deinen vater lebt, kannst du firma fü...","So lange dein Vater lebt, kannst du die Firma ..."


## Create corrections with modell using Ollama

In [8]:
MODEL = "Text-Tune-Base-v7"
corrections = []

In [9]:
len(corrections)

0

In [10]:
from ollama import chat, ChatResponse
from tqdm import tqdm

for index, row in tqdm(df.iterrows(), total=len(df), desc="Correcting rows"):
    original_text = row['original']
    user_input = row['corrupted']

    response: ChatResponse = chat(model=MODEL, messages=[
                            {
                                'role': 'user',
                                'content': user_input,
                            },
                    ])
                    
    corrected_text = response["message"]["content"].strip()

    corrections.append({'input': user_input, 'model_corrected': corrected_text, 'label': original_text, })

Correcting rows: 100%|██████████| 533/533 [03:33<00:00,  2.50it/s]


In [ ]:
pd.DataFrame(corrections).to_json("../outputs/Text-Tune-Base-v7-NEW-DS.jsonl", orient="records", lines=True, force_ascii=False)

## Add Similarity Scores

In [18]:
import ollama
import numpy as np

EMBEDDING_MODEL = "embeddinggemma" #"bge-m3" #"nomic-embed-text"


def get_embeddings(sentences: list[str], model=EMBEDDING_MODEL, show_progress=True) -> np.ndarray:
    """
    Convert a list of sentences into vector embeddings using Ollama.
    """
    embeddings = []
    for sentence in tqdm(sentences, desc="Getting embeddings", disable=not show_progress):
        response = ollama.embed(model=model, input=sentence)
        # print(response)
        embeddings.append(response["embeddings"][0])
    return np.array(embeddings)


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Compute cosine similarity between corresponding rows."""
    # Normalize vectors
    a_norm = a / np.linalg.norm(a, axis=1, keepdims=True)
    b_norm = b / np.linalg.norm(b, axis=1, keepdims=True)
    # Compute dot product for each pair
    return np.sum(a_norm * b_norm, axis=1)


def get_similarity_score(text1: list[str], text2: list[str], model=EMBEDDING_MODEL, show_progress=False) -> list[float]:
    """
    Get similarity score between two texts using Ollama embeddings.
    """
    emb1 = get_embeddings(text1, model=model, show_progress=show_progress)
    emb2 = get_embeddings(text2, model=model, show_progress=show_progress)
    sim = cosine_similarity(emb1, emb2)
    return sim.tolist()


In [19]:
# from src.dpo.ollama_similarity_score import get_similarity_score
corrections_df = pd.read_json("../outputs/Text-Tune-Base-v7-NEW-DS.jsonl", lines=True)
corrections = corrections_df.to_dict(orient="records")

for i in tqdm(range(0, len(corrections), 50), desc="Calculating similarity scores"):
    batch = corrections[i:i+50]
    ground_truths = [item["label"] for item in batch]
    model_outputs = [item["model_corrected"] for item in batch]
    
    scores = get_similarity_score(ground_truths, model_outputs, show_progress=False)
    
    for item, score in zip(batch, scores):
        item["similarity_score"] = score

Calculating similarity scores: 100%|██████████| 132/132 [24:26<00:00, 11.11s/it]


In [20]:
item

{'input': 'Im Mai bestätigte der Landgericht münchen I Verbot.',
 'model_corrected': 'Im Mai bestätigte das Landgericht München I das Verbot.',
 'label': 'Im Mai bestätigte das Landgericht München I das Verbot.',
 'similarity_score': 1.0}

## Construct DPO Pairs

In [21]:
dpo_pairs = []
success_count = 0

In [22]:
from src.prompts import get_inference_prompt_v5


for item in tqdm(corrections, desc="Constructing DPO pairs"):
    input_text = item["input"]
    ground_truth = item["label"]
    model_output = item["model_corrected"]
    score = item["similarity_score"]

    full_prompt = get_inference_prompt_v5(input_text)

    if model_output == ground_truth:
        success_count += 1
        continue
        
    # LAZINESS: The model just copied the input.
    if model_output.strip() == input_text.strip() and ground_truth.strip() != input_text.strip():
        dpo_pairs.append({
            "type": "laziness",
            "prompt": full_prompt, 
            "chosen": ground_truth, 
            "rejected": model_output
        })

    # SIMILARITY: The model output is very similar to the input, but the label is different.
    if score < 0.95:
        dpo_pairs.append({
            "type": f"sim-{score:.2f}",
            "prompt": full_prompt,
            "chosen": ground_truth,
            "rejected": model_output
        })


Constructing DPO pairs: 100%|██████████| 6555/6555 [00:00<00:00, 1157481.70it/s]


In [25]:
len(dpo_pairs)

347

In [26]:
OUTPUT_FILE = "../data/processed/dpo_pairs.jsonl"

pd.DataFrame(dpo_pairs).to_json(OUTPUT_FILE, orient="records", lines=True, force_ascii=False)
print(f"Total pairs constructed for DPO: {len(dpo_pairs)}")
print(f"Total cases where model output was correct (skipped for DPO): {success_count}")

Total pairs constructed for DPO: 347
Total cases where model output was correct (skipped for DPO): 3591
